# Employee Burnout Prediction

Notebook ini dibuat untuk project Data Mining dengan framework **CRISP-DM**.

Versi notebook ini dibuat lebih sederhana dan beginner friendly supaya mudah dipahami saat belajar konsep dasar.

Dataset yang digunakan:

`data/raw/employee_burnout_analysis.csv`

## Framework CRISP-DM

Tahapan yang digunakan pada notebook ini:

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Data Clustering
5. Regression
6. Classification
7. Evaluation
8. Deployment sederhana

## 1. Business Understanding

Tujuan project ini adalah mengelompokkan karyawan berdasarkan risiko burnout.

Hasil clustering akan digunakan sebagai target untuk model regression dan classification.

Output akhir yang ingin didapatkan adalah:

- Predicted cluster
- Low Burnout Risk atau High Burnout Risk
- Recommendation sederhana

## Import Library

Library yang digunakan masih library dasar untuk data mining: pandas, numpy, matplotlib, seaborn, scikit-learn, dan joblib.

In [ ]:
import os
from pathlib import Path

# Cache Matplotlib disimpan di folder project agar aman saat dijalankan lokal.
matplotlib_cache = Path.cwd() / "assets" / "matplotlib_cache"
matplotlib_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_cache))
os.environ.setdefault("XDG_CACHE_HOME", str(matplotlib_cache))
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

## Menyiapkan Path Folder

Kode di bawah menggunakan `pathlib` supaya path aman digunakan di Windows, Mac, Linux, dan Google Colab.

Jika dijalankan di Google Colab, pastikan file dataset sudah ada di folder yang sesuai atau ubah nilai `DATA_PATH`.

In [ ]:
# Jika menggunakan Google Colab dan dataset ada di Google Drive, aktifkan baris berikut.
# from google.colab import drive
# drive.mount('/content/drive')

candidate_base_dirs = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/Tubes Datmin"),
    Path("/content/drive/MyDrive/Tubes Datmin")
]

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "raw" / "employee_burnout_analysis.csv"

for folder in candidate_base_dirs:
    candidate_path = folder / "data" / "raw" / "employee_burnout_analysis.csv"
    if candidate_path.exists():
        BASE_DIR = folder
        DATA_PATH = candidate_path
        break

PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "reports"
MODELS_DIR = BASE_DIR / "models"
ASSETS_DIR = BASE_DIR / "assets"

for folder in [PROCESSED_DIR, REPORTS_DIR, MODELS_DIR, ASSETS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Base folder:", BASE_DIR)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

if not DATA_PATH.exists():
    print("Dataset belum ditemukan. Pastikan file ada di data/raw/employee_burnout_analysis.csv")

## 2. Data Understanding

Pada tahap ini kita membaca dataset, melihat isi data, nama variabel, jumlah record, jumlah variabel, tipe data, sifat data, dan missing values.

In [ ]:
# Dataset menggunakan pemisah titik koma (;) dan angka desimal menggunakan koma (,).
df = pd.read_csv(DATA_PATH, sep=";", decimal=",", encoding="utf-8-sig")

df.head()

In [ ]:
print("Nama variabel dataset:")
print(list(df.columns))

print("\nJumlah variabel:", df.shape[1])
print("Jumlah record:", df.shape[0])

In [ ]:
def get_data_nature(column_name, series):
    if "date" in column_name.lower():
        return "date"
    elif pd.api.types.is_numeric_dtype(series):
        return "numeric"
    elif series.nunique() > 30:
        return "string"
    else:
        return "categorical"

# Laporan sederhana untuk data identification.
data_identification = pd.DataFrame({
    "Variable Name": df.columns,
    "Data Type": [str(df[col].dtype) for col in df.columns],
    "Data Nature": [get_data_nature(col, df[col]) for col in df.columns],
    "Missing Values": [df[col].isna().sum() for col in df.columns],
    "Unique Values": [df[col].nunique() for col in df.columns]
})

data_identification

In [ ]:
# Simpan laporan identifikasi data.
data_identification.to_csv(REPORTS_DIR / "data_identification_report.csv", index=False)

print("Laporan data identification berhasil disimpan.")

## 3. Data Preparation

Pada tahap ini dilakukan beberapa proses sederhana:

- Mengubah kolom tanggal menjadi tipe date
- Mengubah kolom angka menjadi numeric
- Membuat variabel baru `Years Working`
- Mengisi missing values
- Memilih minimal 4 variabel numeric untuk modeling

In [ ]:
df_clean = df.copy()

# Ubah Date of Joining menjadi format tanggal.
df_clean["Date of Joining"] = pd.to_datetime(
    df_clean["Date of Joining"],
    format="%d/%m/%y",
    errors="coerce"
)

# Ubah kolom numerik agar bisa diproses machine learning.
numeric_columns = [
    "Designation",
    "Resource Allocation",
    "Mental Fatigue Score",
    "Burn Rate"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Membuat kolom Years Working dari Date of Joining.
max_year = df_clean["Date of Joining"].dt.year.max()
df_clean["Years Working"] = max_year - df_clean["Date of Joining"].dt.year

df_clean.head()

In [ ]:
# Cek missing values setelah konversi data.
df_clean.isna().sum()

In [ ]:
# Pilih variabel yang akan digunakan untuk clustering, regression, dan classification.
# Di sini kita menggunakan minimal 4 variabel numeric.
features = [
    "Designation",
    "Resource Allocation",
    "Mental Fatigue Score",
    "Years Working"
]

target_reference = "Burn Rate"

print("Fitur numeric yang digunakan:", features)

In [ ]:
# Isi missing values numeric dengan median.
for col in features + [target_reference]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Isi missing values categorical dengan modus.
categorical_columns = ["Gender", "Company Type", "WFH Setup Available"]

for col in categorical_columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

df_clean.isna().sum()

In [ ]:
# Simpan dataset yang sudah dibersihkan.
cleaned_path = PROCESSED_DIR / "cleaned_dataset.csv"
df_clean.to_csv(cleaned_path, index=False)

print("Cleaned dataset disimpan ke:", cleaned_path)

## Visualisasi Sederhana

Sebelum modeling, kita lihat hubungan antar variabel numerik menggunakan heatmap.

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df_clean[features + [target_reference]].corr(), annot=True, cmap="viridis")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "heatmap_notebook.png", dpi=150)
plt.show()

## 4. Data Clustering

Pada tahap ini kita menggunakan **K-Means** dengan jumlah cluster wajib **2**.

Hasil cluster akan ditambahkan ke dataset sebagai kolom `Cluster`.

In [ ]:
X = df_clean[features]

# Scaling diperlukan supaya semua fitur berada pada skala yang seimbang.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-Means dengan 2 cluster.
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df_clean["Cluster"] = kmeans.fit_predict(X_scaled)

df_clean[["Designation", "Resource Allocation", "Mental Fatigue Score", "Years Working", "Burn Rate", "Cluster"]].head()

In [ ]:
# Simpan hasil clustering.
clustering_path = REPORTS_DIR / "clustering_result.csv"
df_clean.to_csv(clustering_path, index=False)

print("Hasil clustering disimpan ke:", clustering_path)

In [ ]:
# Melihat rata-rata Burn Rate setiap cluster.
cluster_summary = df_clean.groupby("Cluster")[["Burn Rate", "Mental Fatigue Score"]].mean()
cluster_summary

In [ ]:
# Cluster dengan rata-rata Burn Rate lebih tinggi dianggap High Burnout Risk.
sorted_clusters = cluster_summary.sort_values("Burn Rate").index.tolist()

risk_label = {
    sorted_clusters[0]: "Low Burnout Risk",
    sorted_clusters[1]: "High Burnout Risk"
}

risk_label

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df_clean,
    x="Resource Allocation",
    y="Mental Fatigue Score",
    hue="Cluster",
    palette="Set2"
)
plt.title("K-Means Clustering Result")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "clustering_notebook.png", dpi=150)
plt.show()

## 5. Regression

Requirement project meminta target variabel berasal dari hasil clustering, yaitu kolom `Cluster`.

Pada tahap regression, kita menggunakan `RandomForestRegressor` untuk memprediksi nilai `Cluster`.

In [ ]:
X = df_clean[features]
y = df_clean["Cluster"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

regression_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1)
regression_model.fit(X_train, y_train)

regression_pred = regression_model.predict(X_test)

print("Regression selesai.")

## 6. Classification

Pada tahap classification, kita menggunakan `RandomForestClassifier` untuk memprediksi kelas `Cluster`.

In [ ]:
classification_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
classification_model.fit(X_train, y_train)

classification_pred = classification_model.predict(X_test)

print("Classification selesai.")

## 7. Evaluation

Pada tahap ini kita mengevaluasi hasil regression dan classification.

Regression dievaluasi dengan MAE, MSE, dan R2 Score.

Classification dievaluasi dengan Accuracy, Confusion Matrix, dan Classification Report.

In [ ]:
mae = mean_absolute_error(y_test, regression_pred)
mse = mean_squared_error(y_test, regression_pred)
r2 = r2_score(y_test, regression_pred)

print("Regression Evaluation")
print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)

In [ ]:
accuracy = accuracy_score(y_test, classification_pred)

print("Classification Evaluation")
print("Accuracy:", accuracy)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, classification_pred))
print("\nClassification Report:")
print(classification_report(y_test, classification_pred))

In [ ]:
# Simpan hasil evaluasi ke file txt.
regression_lines = [
    "Regression Evaluation - RandomForestRegressor",
    "Target variable: Cluster",
    f"Mean Absolute Error: {mae}",
    f"Mean Squared Error: {mse}",
    f"R2 Score: {r2}"
]

classification_lines = [
    "Classification Evaluation - RandomForestClassifier",
    "Target variable: Cluster",
    f"Accuracy: {accuracy}",
    "",
    "Confusion Matrix:",
    str(confusion_matrix(y_test, classification_pred)),
    "",
    "Classification Report:",
    classification_report(y_test, classification_pred)
]

(REPORTS_DIR / "regression_evaluation.txt").write_text("\n".join(regression_lines))
(REPORTS_DIR / "classification_evaluation.txt").write_text("\n".join(classification_lines))

print("File evaluation berhasil disimpan.")

## Menyimpan Model

Model disimpan agar bisa digunakan kembali pada aplikasi deployment, misalnya Streamlit.

In [ ]:
joblib.dump(scaler, MODELS_DIR / "scaler.pkl")
joblib.dump(kmeans, MODELS_DIR / "kmeans.pkl")
joblib.dump(regression_model, MODELS_DIR / "regression_model.pkl")
joblib.dump(classification_model, MODELS_DIR / "classification_model.pkl")

print("Model berhasil disimpan ke folder models.")

## 8. Deployment Sederhana

Pada tahap deployment, project utama menggunakan Streamlit.

Di notebook ini kita membuat simulasi fungsi prediksi sederhana. Input yang digunakan sama seperti aplikasi Streamlit:

- Designation
- Resource Allocation
- Mental Fatigue Score
- Years Working

In [ ]:
def predict_burnout_risk(designation, resource_allocation, mental_fatigue_score, years_working):
    input_data = pd.DataFrame([{
        "Designation": designation,
        "Resource Allocation": resource_allocation,
        "Mental Fatigue Score": mental_fatigue_score,
        "Years Working": years_working
    }])

    predicted_cluster = int(classification_model.predict(input_data)[0])
    risk = risk_label[predicted_cluster]

    if risk == "High Burnout Risk":
        recommendation = "Kurangi beban kerja, perbaiki waktu istirahat, dan lakukan monitoring fatigue."
    else:
        recommendation = "Pertahankan pola kerja saat ini dan tetap pantau kondisi mental fatigue."

    return predicted_cluster, risk, recommendation

In [ ]:
# Contoh prediksi. Silakan ubah nilainya untuk mencoba skenario lain.
cluster, risk, recommendation = predict_burnout_risk(
    designation=2,
    resource_allocation=5,
    mental_fatigue_score=6.5,
    years_working=3
)

print("Predicted cluster:", cluster)
print("Burnout risk:", risk)
print("Recommendation:", recommendation)

## Kesimpulan

Berdasarkan tahapan CRISP-DM, project ini berhasil:

- Membaca dan memahami dataset employee burnout
- Melakukan data preparation
- Membuat clustering dengan K-Means sebanyak 2 cluster
- Menggunakan hasil clustering sebagai target regression dan classification
- Mengevaluasi model
- Membuat simulasi deployment sederhana untuk prediksi risiko burnout